In [51]:
import requests, json, time, socket, urllib3
from confluent_kafka import Producer
urllib3.disable_warnings()

BOOTSTRAP = '172.27.110.36:9092'
TOPIC = 'coingecko_volumes'

def get_exchanges():
    return requests.get("https://api.coingecko.com/api/v3/exchanges/list", verify=False).json()

def volume_generator(exchanges, limit=None):
    count = 0
    for ex in exchanges:
        if limit and count >= limit:
            break
        try:
            url = f"https://api.coingecko.com/api/v3/exchanges/{ex['id']}/volume_chart"
            data = requests.get(url, params={"days": "1"}, verify=False).json()
            
            # Пропускаем, если данных мало или они пустые
            if not isinstance(data, list) or len(data) < 10:
                print(f"Пропуск (мало данных): {ex['name']}")
                continue
                
            # Проверяем, есть ли хоть какой-то объём
            max_vol = max([float(x[1]) for x in data if isinstance(x, list) and len(x) > 1], default=0)
            if max_vol < 0.01:
                print(f"Пропуск (почти нулевой объём): {ex['name']}")
                continue
            
            payload = {ex['name']: data}
            yield json.dumps(payload).encode('utf-8')
            count += 1
            print(f"Отправлено: {ex['name']} (max volume: {max_vol:.2f})")
            time.sleep(1.5)
            
        except Exception as e:
            print(f"Ошибка {ex['name']}: {e}")
            continue

producer = Producer({
    'bootstrap.servers': BOOTSTRAP,
    'client.id': socket.gethostname()
})

exchanges = get_exchanges()
print(f"Найдено бирж: {len(exchanges)}. Начинаю отправку...")

for i, msg in enumerate(volume_generator(exchanges, limit=20), 1):  # 20 бирж для теста
    producer.produce(TOPIC, value=msg)
    print(f"Отправлено {i}")
    
producer.flush()
print("Готово")

Найдено бирж: 1504. Начинаю отправку...
Отправлено 1
Отправлено: 10KSwap (max volume: 0.08)
Отправлено 2
Отправлено: 9inch (max volume: 0.69)
Отправлено 3
Отправлено: 9mm V3 (Pulsechain) (max volume: 12.51)


KeyboardInterrupt: 

In [49]:
from confluent_kafka import Producer
p = Producer({'bootstrap.servers': '172.27.110.36:9092'})
p.produce('coingecko_volumes', b'{"ok":1}')
p.flush()
print('sent')

sent
